# Train a configurable FNO with Hugging Face Accelerate

This notebook trains `FNO3d` from scratch (or resumes a compatible checkpoint) through `scripts/train_fno.py`. Edit **only the CONFIG cell**. The defaults reproduce the architecture of `sim_real_fno.pth`: four layers, width 64, and Fourier modes `(4, 12, 16)`.

Training uses trajectory-level train/validation splits and separate per-split normalization statistics. Checkpoints, optimizer state, architecture configuration, and normalization statistics are written to `SAVE_DIR`.


In [ ]:
# --- Clone or pull repo (public, no token) ---
REPO_DIR = '/kaggle/working/realpde'
REPO_URL = 'https://github.com/nthday-jpg/realpde.git'

!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} pull --ff-only; else git clone {REPO_URL} {REPO_DIR}; fi


In [ ]:
%cd {REPO_DIR}


In [ ]:
# Enforce the project Python version for command-line tools.
!uv python pin 3.10 2>&1 | tail -1
!uv sync --python 3.10


In [ ]:
# Install the source tree into the active notebook kernel.
import os, sys
_REPO = os.environ.get('REPO_DIR', '/kaggle/working/realpde')
!{sys.executable} -m pip install -q -e . --no-deps --ignore-requires-python
if f'{_REPO}/src' not in sys.path:
    sys.path.insert(0, f'{_REPO}/src')


In [ ]:
# --- GPU check ---
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


In [ ]:
# =============================================================================
# CONFIG — edit this cell only. scripts/train_fno.py reads these environment vars.
# =============================================================================
import os
from pathlib import Path

# Authenticate W&B without exposing the key in notebook output. Add WANDB_KEY
# under Kaggle > Add-ons > Secrets and grant this notebook access.
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret('WANDB_KEY')
    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_MODE'] = 'online'  # clear a stale disabled mode
    print('W&B authentication configured from Kaggle Secrets (online mode).')
except Exception as exc:
    # Keep local/offline notebook execution usable without Kaggle Secrets.
    os.environ.setdefault('WANDB_MODE', 'disabled')
    print(f'W&B disabled: WANDB_KEY unavailable ({exc})')

DATA_ROOT = Path('/kaggle/input/datasets/nthday/realpde')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data')

def resolve_h5_dir(path: Path) -> Path:
    if path.exists() and any(path.glob('*.h5')):
        return path
    if path.exists():
        candidates = [p for p in path.rglob('*') if p.is_dir() and any(p.glob('*.h5'))]
        if candidates:
            return candidates[0]
    raise FileNotFoundError(f'No .h5 files found under {path}')

TRAIN_DATA = resolve_h5_dir(DATA_ROOT / 'train_sim')

# Use setdefault so values supplied before this cell remain effective.
os.environ.setdefault('DATA_PATH', str(TRAIN_DATA))
os.environ.setdefault('DATA_CACHE', '')  # optional cache from scripts/cache_dataset.py
os.environ.setdefault('SAVE_DIR', '/kaggle/working/fno_checkpoints')
os.environ.setdefault('RESUME_CKPT', '')  # blank = train from scratch

# Dataset/window configuration
os.environ.setdefault('IN_STEP', '20')
os.environ.setdefault('OUT_STEP', '20')
os.environ.setdefault('INTERVAL', '20')
os.environ.setdefault('SUB_S', '2')
os.environ.setdefault('VAL_FRAC', '0.1')
os.environ.setdefault('SEED', '42')

# FNO architecture (baseline-compatible defaults)
os.environ.setdefault('FNO_MODES1', '4')   # temporal modes
os.environ.setdefault('FNO_MODES2', '12')  # height modes
os.environ.setdefault('FNO_MODES3', '16')  # width/rFFT modes
os.environ.setdefault('FNO_N_LAYERS', '4')
os.environ.setdefault('FNO_WIDTH', '64')
os.environ.setdefault('FNO_PADDING', '6')

# Optimization
os.environ.setdefault('EPOCHS', '50')
os.environ.setdefault('BATCH_SIZE', '1')   # raise only if GPU memory permits
os.environ.setdefault('GRAD_ACCUM_STEPS', '4')
os.environ.setdefault('LR', '1e-4')
os.environ.setdefault('WEIGHT_DECAY', '0')
os.environ.setdefault('MAX_GRAD_NORM', '1.0')
os.environ['MIXED_PRECISION'] = 'no'  # use 'fp16' only with the training-side safe FFT path
os.environ.setdefault('NUM_WORKERS', '4')
# One process per GPU, matching continue_cno_kaggle.ipynb. Set to 1 to
# troubleshoot a device independently. The cache is memory-mapped by the trainer.
os.environ.setdefault('NUM_PROCESSES', str(max(torch.cuda.device_count(), 1)))

# Checkpointing and optional experiment tracking
os.environ.setdefault('SAVE_EVERY', '5')
os.environ.setdefault('SAVE_OPTIMIZER', '1')  # optimizer makes each FNO checkpoint much larger
os.environ.setdefault('LOG_EVERY_STEPS', '10')
# Use assignment, not setdefault: an earlier notebook run may have left these blank.
os.environ['WANDB_PROJECT'] = 'realpde-pretrain'
os.environ['WANDB_RUN_NAME'] = 'fno-sim-scratch'

for key in [
    'DATA_PATH', 'SAVE_DIR', 'RESUME_CKPT', 'FNO_MODES1', 'FNO_MODES2',
    'FNO_MODES3', 'FNO_N_LAYERS', 'FNO_WIDTH', 'FNO_PADDING', 'EPOCHS',
    'BATCH_SIZE', 'GRAD_ACCUM_STEPS', 'LR', 'MIXED_PRECISION', 'NUM_PROCESSES',
    'WANDB_PROJECT', 'WANDB_RUN_NAME', 'WANDB_MODE', 'LOG_EVERY_STEPS'
]:
    print(f'{key:20s} = {os.environ[key]}')


## Optional dataset cache

`PDEDataset` preloads all HDF5 trajectories on every run. For repeated runs, create a cache once with `scripts/cache_dataset.py`, then set `DATA_CACHE` in the CONFIG cell. The cache must use the same window and subsampling settings.


In [ ]:
# OPTIONAL — uncomment once to build a reusable cache.
# cache_path = '/kaggle/working/cache_train_sim.pt'
# cache_workers = os.cpu_count() or 4  # process-based HDF5 loading
# !DATA_PATH="{os.environ['DATA_PATH']}" CACHE_OUT="{cache_path}" CACHE_WORKERS={cache_workers} IN_STEP={os.environ['IN_STEP']} OUT_STEP={os.environ['OUT_STEP']} INTERVAL={os.environ['INTERVAL']} SUB_S={os.environ['SUB_S']} {sys.executable} scripts/cache_dataset.py
# os.environ['DATA_CACHE'] = cache_path


## Launch training

Accelerate starts one process per visible GPU. `BATCH_SIZE` is per process; the effective batch size is `BATCH_SIZE × NUM_PROCESSES × GRAD_ACCUM_STEPS`. The default uses all visible GPUs. The trainer memory-maps the approximately 5 GB cache so ranks can share the operating-system page cache.

For a fresh run, leave `RESUME_CKPT` blank. To resume, point it at `last.pth` or `best.pth` and keep the FNO architecture settings identical to the checkpoint.


In [ ]:
# --- Train with Hugging Face Accelerate ---
# Use the project Python 3.10 environment, matching continue_cno_kaggle.ipynb.
import os, torch
num_processes = int(os.environ['NUM_PROCESSES'])
if num_processes > max(torch.cuda.device_count(), 1):
    raise ValueError(f'NUM_PROCESSES={num_processes} exceeds visible devices')
mixed_precision = os.environ['MIXED_PRECISION'].strip().lower()
# Override any stale Accelerate state/config from an earlier fp16 notebook run.
os.environ['ACCELERATE_MIXED_PRECISION'] = mixed_precision
distributed_flag = '--multi_gpu' if num_processes > 1 else ''
print(f'Launching FNO training with {num_processes} process(es), precision={mixed_precision}')

!uv run --python 3.10 accelerate launch {distributed_flag} --mixed_precision={mixed_precision} --num_processes={num_processes} --num_machines=1 --dynamo_backend=no scripts/train_fno.py


In [ ]:
# --- Inspect outputs and saved configuration ---
from pathlib import Path
import os, torch

save_dir = Path(os.environ['SAVE_DIR'])
for path in sorted(save_dir.glob('*')):
    print(f'{path.name:24s} {path.stat().st_size / 2**20:9.1f} MiB')

best = save_dir / 'best.pth'
if best.exists():
    checkpoint = torch.load(best, map_location='cpu', weights_only=False)
    print()
    print('Best epoch:', checkpoint.get('epoch'))
    print('Validation MSE:', checkpoint.get('val_loss'))
    print('FNO config:', {k: checkpoint['cfg'][k] for k in
          ('modes1', 'modes2', 'modes3', 'n_layers', 'width', 'padding')})


## Notes

- The baseline-sized FNO has roughly 50 million logical parameter elements and large complex spectral weights. Its FP32 model checkpoint is about 385 MiB; checkpoints containing Adam optimizer state are substantially larger.
- If CUDA runs out of memory, keep `BATCH_SIZE=1`, increase `GRAD_ACCUM_STEPS`, or reduce `FNO_WIDTH`/Fourier modes.
- `best.pth` and `last.pth` use the standard `model_state_dict` format and include `cfg`, `norm_train`, and `norm_val` for reproducibility.
